# Stage 1 – Subscription Prediction
Owner: _name_

**Rules followed here:** the test set is only used once, at the very end, to predict. All preprocessing (scaling, one-hot encoding, PCA, feature selection) is fit inside each CV fold on training rows only. Model code lives in `src/stage1.py`.

In [1]:
import sys, warnings; sys.path.append('../src'); warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from config import SEED, STAGE1_RESULTS
from stage1 import load_stage1, classical_model, QuantumKernelClassifier, cross_validate
np.random.seed(SEED)
train, test, X, y = load_stage1()
print(f'train {X.shape}, test {test.shape}, positives {y.sum()} ({y.mean():.1%})')

train (300, 15), test (100, 16), positives 35 (11.7%)


## 1. Why accuracy is useless here
Only ~12% of customers subscribe, so "always no" scores 88% accuracy. We judge with ROC-AUC, PR-AUC (random ≈ 0.12), Precision/Recall@10, and probability quality (Brier, log loss). For reference, always predicting the base rate gives Brier ≈ 0.103.

In [2]:
base = y.mean(); print('Brier of predicting the base rate for everyone:', round(base*(1-base), 4))

Brier of predicting the base rate for everyone: 0.1031


## 2. Classical baseline – logistic regression
Preprocessing: `pdays = -1` becomes a `was_contacted_before` flag, balance is log-scaled, numerics standardized, categoricals one-hot.

We compare unweighted vs `class_weight='balanced'`. Balanced weighting can help ranking but **inflates probabilities**, which hurts Brier score and would distort Stage 2 expected values (p × deposit − cost) and Stage 3. So we keep probabilities unweighted.

In [3]:
results = {}
for name, m in {'logreg': classical_model(C=0.3), 'logreg_balanced': classical_model(C=0.3, class_weight='balanced')}.items():
    folds, _ = cross_validate(m, X, y)
    results[name] = folds.drop(columns='fold').mean()
pd.DataFrame(results).T.round(3)

,roc_auc,pr_auc,brier,log_loss,precision_at_10,recall_at_10
logreg,0.648,0.284,0.101,0.351,0.16,0.229
logreg_balanced,0.657,0.265,0.202,0.589,0.20,0.286


## 3. Quantum classifier – quantum kernel SVM
Pipeline: preprocessing → reduce to *n* features (one per qubit) → angle scaling → **ZZFeatureMap** encoding → fidelity kernel K(x, x') = |⟨φ(x)|φ(x')⟩|² computed exactly on a statevector simulator → SVC on the precomputed kernel, with Platt scaling for probabilities.

What is quantum: only the kernel (the similarity between customers). Everything else is classical.

Two ways to reduce features, both fit inside each fold: PCA (unsupervised) and L1-logistic selection (supervised). `scale` controls kernel width: too large and every customer looks unrelated to every other (the kernel ≈ identity).

In [4]:
grid = []
for reducer in ['pca', 'select']:
    for n_qubits in [4, 6]:
        for scale in [0.15, 0.3, 0.6]:
            folds, _ = cross_validate(QuantumKernelClassifier(n_qubits, 1, 1.0, scale, reducer), X, y)
            grid.append({'reducer': reducer, 'qubits': n_qubits, 'scale': scale, **folds.drop(columns='fold').mean().to_dict()})
pd.DataFrame(grid).round(3).sort_values('roc_auc', ascending=False)

,reducer,qubits,scale,roc_auc,pr_auc,brier,log_loss,precision_at_10,recall_at_10
9,select,6,0.15,0.632,0.252,0.102,0.355,0.22,0.314
5,pca,6,0.60,0.594,0.230,0.102,0.355,0.20,0.286
6,select,4,0.15,0.562,0.204,0.102,0.367,0.20,0.286
4,pca,6,0.30,0.554,0.178,0.103,0.358,0.18,0.257
7,select,4,0.30,0.538,0.186,0.102,0.366,0.18,0.257
3,pca,6,0.15,0.534,0.183,0.104,0.364,0.14,0.200
10,select,6,0.30,0.521,0.188,0.104,0.362,0.12,0.171
8,select,4,0.60,0.479,0.188,0.102,0.360,0.16,0.229
2,pca,4,0.60,0.474,0.130,0.104,0.363,0.06,0.086
11,select,6,0.60,0.448,0.167,0.104,0.362,0.08,0.114


## 4. Honest comparison with repeated CV
With only 35 positives, a single 5-fold split is noisy, and picking the best grid cell above slightly flatters the quantum model. We re-check the leaders over 5 different CV shuffles (25 folds) and report mean ± std.

In [5]:
candidates = {
    'classical: logreg': classical_model(C=0.3),
    'quantum: kernel (select, 6q, scale 0.15)': QuantumKernelClassifier(6, 1, 1.0, 0.15, 'select'),
    'quantum: kernel (pca, 6q, scale 0.6)': QuantumKernelClassifier(6, 1, 1.0, 0.6, 'pca'),
}
summary = {}
for name, m in candidates.items():
    r = pd.concat([cross_validate(m, X, y, seed=s)[0] for s in range(5)]).drop(columns='fold')
    summary[name] = {k: f'{r[k].mean():.3f} ± {r[k].std():.3f}' for k in ['roc_auc', 'pr_auc', 'brier', 'log_loss', 'precision_at_10', 'recall_at_10']}
pd.DataFrame(summary).T

,roc_auc,pr_auc,brier,log_loss,precision_at_10,recall_at_10
classical: logreg,0.595 ± 0.096,0.254 ± 0.099,0.103 ± 0.007,0.366 ± 0.030,0.192 ± 0.108,0.274 ± 0.154
"quantum: kernel (select, 6q, scale 0.15)",0.517 ± 0.121,0.190 ± 0.080,0.105 ± 0.006,0.368 ± 0.021,0.136 ± 0.091,0.194 ± 0.130
"quantum: kernel (pca, 6q, scale 0.6)",0.506 ± 0.128,0.191 ± 0.091,0.104 ± 0.003,0.367 ± 0.015,0.144 ± 0.100,0.206 ± 0.143


**Reading the results:** the classical logistic regression ranks customers better than the quantum kernel, and both beat random (ROC-AUC 0.5, PR-AUC ≈ 0.12) only modestly. The data is small and the signal is weak (`duration`, the strongest feature in the original dataset, was removed). We make **no quantum-advantage claim**. The quantum model is a working, fairly tuned hybrid alternative.

Final model for the pipeline: **logistic regression**, chosen on CV performance alone (the test set was never used).

## 5. Fit the final model on all training data and predict the test set once

In [6]:
final = classical_model(C=0.3).fit(X, y)
test_probs = final.predict_proba(test[X.columns])[:, 1]

# also keep quantum predictions for the report (not used downstream)
qk = QuantumKernelClassifier(6, 1, 1.0, 0.15, 'select').fit(X, y)
test_probs_quantum = qk.predict_proba(test[X.columns])[:, 1]

## 6. Rank (ties → lower customer_id first) and save the handoff file

In [7]:
out = pd.DataFrame({'customer_id': test['customer_id'], 'predicted_probability': test_probs,
                    'quantum_probability': test_probs_quantum})
out = out.sort_values(['predicted_probability', 'customer_id'], ascending=[False, True]).reset_index(drop=True)
out['rank'] = np.arange(1, len(out) + 1)
out['is_top_10'] = out['rank'] <= 10
out.to_csv(STAGE1_RESULTS, index=False)
out.head(12)

,customer_id,predicted_probability,quantum_probability,rank,is_top_10
0,C0031,0.428269,0.788142,1,True
1,C0079,0.384625,0.109952,2,True
2,C0183,0.281598,0.107951,3,True
3,C0081,0.262170,0.113108,4,True
4,C0318,0.249147,0.107951,5,True
5,C0208,0.232900,0.107951,6,True
6,C0338,0.218127,0.107992,7,True
7,C0059,0.206004,0.107951,8,True
8,C0265,0.205985,0.108007,9,True
9,C0170,0.197733,0.108007,10,True
